# Time-Gated Multi-Lingual Time Series Analysis
## Google Play Store — App Installs by Category (MoM Growth)


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import matplotlib.dates as mdates
import seaborn as sns
from datetime import datetime
import pytz
import warnings
warnings.filterwarnings('ignore')


In [ ]:
apps_df    = pd.read_csv('Play Store Data.csv')
reviews_df = pd.read_csv('User Reviews.csv')

apps_df.head()


In [ ]:
apps_df.dtypes


In [ ]:
apps_df.dropna(subset=['App', 'Category', 'Installs', 'Reviews', 'Last Updated'], inplace=True)
print(f"Rows after null removal: {len(apps_df)}")


In [ ]:
apps_df['Installs'] = (
    apps_df['Installs']
    .astype(str)
    .str.replace(',', '', regex=False)
    .str.replace('+', '', regex=False)
    .str.strip()
)
apps_df['Installs'] = pd.to_numeric(apps_df['Installs'], errors='coerce')
apps_df.dropna(subset=['Installs'], inplace=True)
apps_df['Installs'] = apps_df['Installs'].astype(int)
print("Sample Installs:", apps_df['Installs'].head().tolist())


In [ ]:
apps_df['Reviews'] = pd.to_numeric(apps_df['Reviews'], errors='coerce')
apps_df.dropna(subset=['Reviews'], inplace=True)
apps_df['Reviews'] = apps_df['Reviews'].astype(int)

apps_df['Date'] = pd.to_datetime(apps_df['Last Updated'], errors='coerce')
apps_df.dropna(subset=['Date'], inplace=True)

print(f"Rows after type cleaning: {len(apps_df)}")


In [ ]:
ist     = pytz.timezone('Asia/Kolkata')
now_ist = datetime.now(ist)
hour    = now_ist.hour + now_ist.minute / 60

in_window = 18.0 <= hour < 21.0
print(f"Current IST: {now_ist.strftime('%I:%M %p')} | Chart visible: {in_window}")


In [ ]:
if not in_window:
    fig, ax = plt.subplots(figsize=(10, 3))
    fig.patch.set_facecolor('#0d1117')
    ax.set_facecolor('#0d1117')
    ax.axis('off')
    ax.text(0.5, 0.6, "🔒  Access Restricted",
            transform=ax.transAxes, ha='center', fontsize=20,
            fontweight='bold', color='#ff6b6b')
    ax.text(0.5, 0.3, "This visualization is exclusive and only accessible between 6:00 PM and 9:00 PM IST.",
            transform=ax.transAxes, ha='center', fontsize=12, color='#e0e0e0', style='italic')
    plt.tight_layout()
    plt.show()
    raise SystemExit("Outside 6 PM–9 PM IST window.")


In [ ]:
# Rule 1 — Reviews > 500
apps_df = apps_df[apps_df['Reviews'] > 500]
print(f"After Reviews > 500: {len(apps_df)}")


In [ ]:
# Rule 2 — Drop apps starting with X, Y, Z
apps_df = apps_df[~apps_df['App'].str[0].str.upper().isin(['X', 'Y', 'Z'])]
print(f"After dropping X/Y/Z apps: {len(apps_df)}")


In [ ]:
# Rule 3 — Drop apps containing the letter 'S'
apps_df = apps_df[~apps_df['App'].str.upper().str.contains('S', regex=False)]
print(f"After dropping apps with 'S': {len(apps_df)}")


In [ ]:
# Rule 4 — Category starts with E, C, B  OR  is Dating
cat_upper = apps_df['Category'].str.upper()
apps_df   = apps_df[cat_upper.str[0].isin(['E', 'C', 'B']) | (cat_upper == 'DATING')]
print(f"After category filter: {len(apps_df)}")
print(f"Categories: {sorted(apps_df['Category'].unique())}")


In [ ]:
locale_map = {
    'BEAUTY'   : 'सौंदर्य',       # Hindi
    'BUSINESS' : 'வணிகம்',        # Tamil
    'DATING'   : 'Partnersuche',  # German
}

apps_df['Category_Label'] = apps_df['Category'].apply(
    lambda c: locale_map.get(c.upper(), c.replace('_', ' ').title())
)

apps_df[['Category', 'Category_Label']].drop_duplicates().sort_values('Category')


In [ ]:
apps_df['Month'] = apps_df['Date'].dt.to_period('M').dt.to_timestamp()

monthly = (
    apps_df.groupby(['Month', 'Category_Label'])['Installs']
    .sum()
    .reset_index()
    .rename(columns={'Installs': 'Total_Installs'})
    .sort_values(['Category_Label', 'Month'])
)

monthly['MoM_Growth'] = monthly.groupby('Category_Label')['Total_Installs'].pct_change()

monthly.head(12)


In [ ]:
MOM_THRESHOLD = 0.20
PALETTE = ['#4C9BE8','#F28C28','#3DBE92','#E85D75','#A78BFA',
           '#F9C846','#60C6C4','#F07070','#86EFAC','#FBA4C0']

categories = sorted(monthly['Category_Label'].unique())
color_map  = {cat: PALETTE[i % len(PALETTE)] for i, cat in enumerate(categories)}

fig, ax = plt.subplots(figsize=(16, 7))
fig.patch.set_facecolor('#0F172A')
ax.set_facecolor('#1E293B')
ax.grid(True, linestyle='--', linewidth=0.5, color='#334155', alpha=0.6)
ax.set_axisbelow(True)

for spine in ax.spines.values():
    spine.set_edgecolor('#334155')
ax.tick_params(colors='#CBD5E1', labelsize=9)

for cat in categories:
    sub   = monthly[monthly['Category_Label'] == cat].sort_values('Month')
    x, y  = sub['Month'].values, sub['Total_Installs'].values
    mom   = sub['MoM_Growth'].values
    color = color_map[cat]

    ax.plot(x, y, color=color, linewidth=2.4, marker='o', markersize=5, label=cat, zorder=3)

    for i in range(1, len(sub)):
        if pd.notna(mom[i]) and mom[i] > MOM_THRESHOLD:
            ax.fill_between([x[i-1], x[i]], [y[i-1], y[i]], alpha=0.20, color=color, zorder=2)
            ax.annotate(f"+{mom[i]*100:.0f}%", xy=(x[i], y[i]),
                        xytext=(4, 6), textcoords='offset points',
                        fontsize=7.5, color=color, fontweight='bold')

ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=2))
ax.yaxis.set_major_formatter(
    ticker.FuncFormatter(lambda v, _: f'{int(v/1e6)}M' if v >= 1e6 else f'{int(v/1e3)}K')
)
plt.xticks(rotation=40, ha='right', fontsize=8.5, color='#CBD5E1')
plt.yticks(fontsize=8.5, color='#CBD5E1')

ax.set_title('Total App Installs Over Time — by Category', fontsize=15,
             fontweight='bold', color='#F1F5F9', pad=14)
ax.set_xlabel('Month', fontsize=11, color='#94A3B8', labelpad=10)
ax.set_ylabel('Total Installs', fontsize=11, color='#94A3B8', labelpad=10)
ax.legend(fontsize=9, framealpha=0.2, facecolor='#1E293B',
          edgecolor='#475569', labelcolor='#E2E8F0', ncol=2)

plt.tight_layout(pad=1.5)
plt.savefig('timeseries_installs_by_category.png', dpi=180, bbox_inches='tight',
            facecolor=fig.get_facecolor())
plt.show()
print("Chart saved → timeseries_installs_by_category.png")


In [ ]:
pivot = monthly.pivot_table(
    index='Category_Label', columns='Month',
    values='MoM_Growth', aggfunc='mean'
)
pivot.columns = [c.strftime('%b %Y') for c in pivot.columns]

fig2, ax2 = plt.subplots(figsize=(16, max(4, len(pivot) * 0.7 + 1)))
fig2.patch.set_facecolor('#0F172A')

sns.heatmap((pivot * 100).round(1), ax=ax2, cmap='RdYlGn', center=0,
            annot=True, fmt='.1f', linewidths=0.4, linecolor='#1E293B',
            cbar_kws={'label': 'MoM Growth (%)', 'shrink': 0.6},
            annot_kws={'size': 8})

ax2.set_title('Month-over-Month Install Growth (%) by Category',
              fontsize=13, fontweight='bold', color='#F1F5F9', pad=14)
ax2.set_xlabel('Month', fontsize=10, color='#94A3B8', labelpad=8)
ax2.set_ylabel('Category', fontsize=10, color='#94A3B8', labelpad=8)
ax2.tick_params(colors='#CBD5E1', labelsize=8.5)
plt.xticks(rotation=40, ha='right')

plt.tight_layout(pad=1.5)
plt.savefig('mom_growth_heatmap.png', dpi=180, bbox_inches='tight',
            facecolor=fig2.get_facecolor())
plt.show()
print("Heatmap saved → mom_growth_heatmap.png")


In [ ]:
summary = (
    monthly.groupby('Category_Label')
    .agg(
        Total_Installs      = ('Total_Installs', 'sum'),
        Avg_Monthly_Installs= ('Total_Installs', 'mean'),
        Avg_MoM_Growth      = ('MoM_Growth', 'mean'),
        Max_MoM_Growth      = ('MoM_Growth', 'max'),
        High_Growth_Months  = ('MoM_Growth', lambda x: (x > MOM_THRESHOLD).sum()),
    )
    .reset_index()
    .sort_values('Total_Installs', ascending=False)
)

summary['Avg_MoM_Growth'] = (summary['Avg_MoM_Growth'] * 100).round(1).astype(str) + '%'
summary['Max_MoM_Growth'] = (summary['Max_MoM_Growth'] * 100).round(1).astype(str) + '%'

print(summary.to_string(index=False))
